# Searching Live News with Hindi Queries using BERT
<br></br>

## Author: Dr Partha Majumdar
#### ORC-ID: 0009-0002-7375-8034

# STEP 1: Install required libraries

In [1]:
# !pip install -qq feedparser sentence-transformers scikit-learn pandas requests

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 2.4 MB/s eta 0:00:00


# STEP 2: Import libraries

In [2]:
import feedparser
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# STEP 3: Define live RSS feeds

In [3]:
rss_feeds = {
    "HT Latest": "https://www.hindustantimes.com/feeds/rss/latest/rssfeed.xml",
    "HT India": "https://www.hindustantimes.com/feeds/rss/india-news/rssfeed.xml",
    "HT World": "https://www.hindustantimes.com/feeds/rss/world-news/rssfeed.xml",
    "HT Technology": "https://www.hindustantimes.com/feeds/rss/technology/rssfeed.xml",
    "HT Business": "https://www.hindustantimes.com/feeds/rss/business/rssfeed.xml"
}

# STEP 4: Collect articles from live RSS feeds

In [4]:
records = []

for source_name, feed_url in rss_feeds.items():
    feed = feedparser.parse(feed_url)

    for entry in feed.entries:
        title = entry.get("title", "").strip()
        summary = entry.get("summary", "").strip()
        link = entry.get("link", "").strip()
        published = entry.get("published", "").strip()

        # Combine title and summary into one searchable document
        document = f"{title}. {summary}".strip()

        if title and document:
            records.append({
                "source": source_name,
                "title": title,
                "summary": summary,
                "published": published,
                "link": link,
                "document": document
            })

df = pd.DataFrame(records)

# Remove duplicates if the same article appears in multiple feeds
df.drop_duplicates(subset=["title", "link"], inplace=True)
df.reset_index(drop=True, inplace=True)

print("Number of live articles collected:", len(df))
print(df[["source", "title"]].head())

Number of live articles collected: 288
      source                                              title
0  HT Latest  What does Trump’s massive $1.5 trillion defens...
1  HT Latest  GT vs MI, IPL 2026: Mumbai Indians cruise past...
2  HT Latest  Joe Rogan can't stop laughing at Trump's 'I th...
3  HT Latest  Did The Onion buy Infowars? Here's what's happ...
4  HT Latest  ‘Under no pressure to make a deal’: Trump lash...


# STEP 5: Load a multilingual BERT-based embedding model

This model can map Hindi and English text into a shared space.

In [5]:
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

print("Multilingual BERT-based embedding model loaded successfully.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Multilingual BERT-based embedding model loaded successfully.


# STEP 6: Create embeddings for all live news articles

In [6]:
documents = df["document"].tolist()

document_embeddings = model.encode(
    documents,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Document embedding shape:", document_embeddings.shape)

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Document embedding shape: (288, 384)


# STEP 7: Define a semantic search function

Hindi query -> most relevant live news articles

In [7]:
def search_live_news(hindi_query, top_k=5):
    query_embedding = model.encode([hindi_query], convert_to_numpy=True)

    scores = cosine_similarity(query_embedding, document_embeddings)[0]
    top_indices = np.argsort(scores)[-top_k:][::-1]

    results = []
    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "rank": rank,
            "score": round(float(scores[idx]), 4),
            "source": df.iloc[idx]["source"],
            "published": df.iloc[idx]["published"],
            "title": df.iloc[idx]["title"],
            "summary": df.iloc[idx]["summary"],
            "link": df.iloc[idx]["link"]
        })

    return pd.DataFrame(results)

# STEP 8: Test with sample Hindi queries

In [12]:
sample_queries = [
    "कृत्रिम बुद्धिमत्ता और नई तकनीक",
    "भारत की राजनीति",
    "शेयर बाजार और अर्थव्यवस्था",
    "खेल और क्रिकेट समाचार",
    "वैश्विक संघर्ष और विश्व समाचार"
]

import textwrap

WIDTH = 60

def wrap(text):
    return textwrap.fill(str(text), width=WIDTH)

def display_results(results, query):
    print("=" * WIDTH)
    print(wrap(f"Hindi Query: {query}"))
    print("=" * WIDTH)

    for _, row in results.iterrows():
        print(wrap(f"Rank      : {row['rank']}"))
        print(wrap(f"Score     : {row['score']}"))
        print(wrap(f"Source    : {row['source']}"))
        print(wrap(f"Published : {row['published']}"))
        print(wrap(f"Title     : {row['title']}"))
        print(wrap(f"Summary   : {row['summary']}"))
        print(wrap(f"Link      : {row['link']}"))
        print("-" * WIDTH)

for query in sample_queries:
    results = search_live_news(query, top_k=3)
    display_results(results, query)

Hindi Query: कृत्रिम बुद्धिमत्ता और नई तकनीक
Rank      : 1
Score     : 0.4789
Source    : HT Technology
Published : Mon, 20 Apr 2026 16:01:07 +0530
Title     : No to laissez-faire on AI, yes to a light touch
Summary   : The private sector can do much of the heavy
lifting in verifying safety claims, writes Dean Ball
Link      : https://www.hindustantimes.com/technology/no-to-
laissez-faire-on-ai-yes-to-a-light-
touch-101776677137474.html
------------------------------------------------------------
Rank      : 2
Score     : 0.2836
Source    : HT Technology
Published : Mon, 20 Apr 2026 18:12:19 +0530
Title     : Samsung Galaxy A57 review: A well-rounded
premium mid-ranger with AI smarts
Summary   : The Samsung Galaxy A57 is a well rounded premium
mid-ranger that strives to be more.
Link      :
https://www.hindustantimes.com/technology/samsung-
galaxy-a57-review-a-well-rounded-premium-mid-ranger-with-ai-
smarts-101776685062151.html
----------------------------------------------------------

The results demonstrate, in a very tangible way, the strength of BERT in capturing meaning across languages and mapping it into a shared semantic space. For the query “कृत्रिम बुद्धिमत्ता और नई तकनीक” (artificial intelligence and new technology), all three retrieved articles belong to the technology domain, and more importantly, they are conceptually aligned with the idea of emerging technologies. The top-ranked article discusses AI regulation, while the others refer to AI-enabled devices and advances in robotics. Even though the query is in Hindi and the articles are in English, the model successfully bridges the linguistic gap and retrieves content based on meaning rather than exact word matching. The moderate similarity scores reflect the nuance of semantic matching—these articles are related to the theme, but not identical in wording.

The query “भारत की राजनीति” (Indian politics) produces a much stronger alignment, as seen from the higher similarity scores. All retrieved articles are clearly rooted in political discourse, including regional politics, electoral dynamics, and governance debates. This indicates that the model performs particularly well when the query corresponds to a well-defined and frequently occurring theme in the dataset. The consistency across the top three results suggests that the embedding space has effectively clustered political content, allowing the query to land in a dense and semantically coherent region.

For “शेयर बाजार और अर्थव्यवस्था” (stock market and economy), the first result is highly relevant, directly discussing stock indices and market movements. However, the second and third results show a slight drift, moving into sports and geopolitical finance respectively. This illustrates an important aspect of semantic search: while the model captures broad contextual similarity, it may occasionally surface results that are only partially related, especially when the dataset contains mixed or overlapping themes. The presence of financial elements in a global conflict article, for instance, is sufficient for it to be considered relevant, though not strictly aligned with the query's intent.

The query “खेल और क्रिकेट समाचार” (sports and cricket news) reveals both the strengths and limitations of the system. While two of the retrieved articles are clearly about cricket, the top-ranked result is unrelated, focusing instead on cultural promotion. This suggests that the embedding model has picked up on a more abstract or tangential association, possibly due to overlapping contextual signals in the training data. It highlights that semantic similarity is not always perfectly aligned with human expectations of topical relevance, especially when the signal for a specific domain like cricket is relatively weaker or diluted in the embedding space.

Finally, the query “वैश्विक संघर्ष और विश्व समाचार” (global conflict and world news) yields highly coherent results. All three articles are from the world news category and deal with geopolitical themes, including international relations, military tensions, and regional security. The scores are relatively close, indicating that multiple articles occupy similar positions in the semantic space. This reinforces the idea that BERT is particularly effective when the query maps onto a well-defined global theme with consistent representation in the data.

Taken together, these results validate the central premise of the application: BERT enables semantic retrieval that goes beyond keywords, connecting queries and documents through meaning. At the same time, the occasional mismatches serve as a reminder that semantic understanding is probabilistic and context-dependent. The system does not “know” topics in a rigid sense; it infers relationships based on learned patterns in language, which may sometimes lead to partial or unexpected associations.

# STEP 9: Save current feed snapshot and embeddings

In [9]:
df.to_csv("live_news_rss_snapshot.csv", index=False)
np.save("live_news_embeddings.npy", document_embeddings)

print("\nSaved:")
print("- live_news_rss_snapshot.csv")
print("- live_news_embeddings.npy")


Saved:
- live_news_rss_snapshot.csv
- live_news_embeddings.npy
